In [1]:
# key imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset
from torchvision import datasets
import torchvision
import torchvision.transforms as transforms
from torchvision.io import read_image 
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import os

# Load Data

In [2]:
# dataset path
dataset_path = '../data/processed'
splits = ['Training', 'Testing', 'Validation']
classes = ['no_tumor', 'glioma_tumor', 'meningioma_tumor', 'pituitary_tumor']

## CSV File

We need a csv file that contains one line for each image: 'path, label'. 

In [3]:
for split in splits:
    image_list = []
    # open a new csv file for each split
    with open(f'../data/processed/mri_dataset_{split}.csv', 'w') as f:

        for cls in classes:
            class_dir = os.path.join(dataset_path, split, cls)
            for img_name in os.listdir(class_dir):
                full_path = os.path.join(class_dir, img_name)
                tuple = (full_path, cls)
                image_list.append(tuple)

        # save to csv
        f.write('image_path,label\n')
        for item in image_list:
            f.write(f'{item[0]},{item[1]}\n')

In [4]:
class custom_dataset(Dataset):
    # defining constructor
    def __init__(self, annotations, directory, transform=None):
        # directory containing the images
        self.directory = directory
        annotations_file_dir = os.path.join(self.directory, annotations)
        # loading the csv with info about images
        self.labels = pd.read_csv(annotations_file_dir)
        # transform to be applied on images
        self.transform = transform
 
        # Number of images in dataset
        self.len = self.labels.shape[0]
 
    # getting the length
    def __len__(self):
        return len(self.labels)
 
    # getting the data items
    def __getitem__(self, idx):
        # defining the image path
        image_path = self.labels.iloc[idx, 0]
        # reading the images
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE) # reads image into a numpy array of shape (H, W) since it's grayscale
        # corresponding class labels of the images 
        label = self.labels.iloc[idx, 1]
 
        # apply the transform if not set to None
        if self.transform:
            image = Image.fromarray(image) # convert numpy array to PIL Image
            image = self.transform(image) # apply the transform to the image, which will convert it to a tensor and normalize it
        
        # returning the image and label
        return image, label

In [5]:
# print type of the custom dataset images
print(type(custom_dataset))

<class 'type'>


## Dataset and Transformations Explanation

The next cell creates the training, testing, and validation datasets for the MRI brain tumor project, using a custom PyTorch Dataset class. Each dataset is wrapped in a DataLoader for batching and shuffling.

### Transformations

- **Training Transform (`transform_train`)**:  
  - `RandomHorizontalFlip()`: Randomly flips images horizontally with a default probability of 0.5.
  - `RandomCrop(32, padding=4)`: Randomly crops images to 32x32 pixels, with up to 4 pixels of padding.
  - `ToTensor()`: Converts PIL images or numpy arrays to PyTorch tensors.
  - `Normalize((0.5,), (0.5,))`: Normalizes the tensor so that pixel values are scaled to have mean 0.5 and standard deviation 0.5 (for grayscale images).

- **Test/Validation Transform (`transform_test`)**:  
  - Only `ToTensor()` and `Normalize((0.5,), (0.5,))` are applied. No augmentation, so images are just converted and normalized.

### Normalization Constants

- The constants `(0.5,)` for mean and std are used to scale pixel values.  
  - After `ToTensor()`, pixel values are in `[0, 1]`.
  - Normalization transforms them to `(value - 0.5) / 0.5`, so values are centered around 0.

### Augmentation

- **RandomHorizontalFlip**: About half of the training images are flipped.
- **RandomCrop**: All training images are cropped (with random position and padding).
- **Test/Validation**: No augmentation, only conversion and normalization.

---

**Summary:**  
- Training set: Augmented (flip/crop), converted, normalized.
- Test/validation: Only converted and normalized.
- Normalization centers pixel values around 0.

In [6]:
transform_train = transforms.Compose([
    transforms.Resize((64, 64)), 
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(64, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

transform_test = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# training dataset
trainset = custom_dataset(annotations='mri_dataset_Training.csv', directory='../data/processed', transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True, num_workers=0)

testset = custom_dataset(annotations='mri_dataset_Testing.csv', directory='../data/processed', transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False, num_workers=0)

valset = custom_dataset(annotations='mri_dataset_Validation.csv', directory='../data/processed', transform=transform_test)
valloader = torch.utils.data.DataLoader(valset, batch_size=32, shuffle=False, num_workers=0)

In [7]:
# display number of images in each split
print(f'Number of training images: {len(trainset)}')
print(f'Number of testing images: {len(testset)}')
print(f'Number of validation images: {len(valset)}')

# display the path and label of the first image in trainset
print(f'Path of the first image in trainset: {trainset.labels.iloc[0, 0]}')
print(f'Label of the first image in trainset: {trainset.labels.iloc[0, 1]}')

Number of training images: 2031
Number of testing images: 587
Number of validation images: 784
Path of the first image in trainset: ../data/processed/Training/no_tumor/image(115).jpg
Label of the first image in trainset: no_tumor


# Load Model

## What is the BasicBlock and Why Use It?

The `BasicBlock` is a fundamental building block of the ResNet architecture. Its main purpose is to allow the network to be much deeper without suffering from the vanishing gradient problem, which can make training deep networks difficult.

### How does it work?
- The block contains two convolutional layers, each followed by batch normalization and ReLU activation.
- It uses a **shortcut connection** (also called a skip connection or residual connection) that adds the input directly to the output of the block.
- If the input and output shapes differ, the shortcut uses a 1x1 convolution to match dimensions.

### Why is this important?
- The shortcut connection lets information and gradients flow more easily through the network.
- This helps prevent the vanishing gradient problem, making it possible to train very deep networks.
- Even if you don't fully understand all the details, using this block is standard practice for modern deep convolutional networks.

**Summary:**  
The BasicBlock enables deep networks to learn effectively by allowing gradients to bypass some layers, making training more stable and efficient.

In [8]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += self.shortcut(x)
        out = self.relu(out)
        return out

In [9]:
class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet18, self).__init__()
        self.in_channels = 64
        # only accept 1 channel input since our images are grayscale
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        
        self.layer1 = self._make_layer(BasicBlock, 64, 2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, 128, 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, 2, stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResNet18(num_classes=len(classes)).to(device)
print(model)

ResNet18(
  (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential()
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

# Initial Performance

In [11]:
# put model into evaluation mode
model.eval()

# define function to predict outcome of a single image
def classify_image(image):
    # If image is a numpy array, convert to PIL Image for transform
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    # if the image is a tensor, convert to PIL Image for transform
    elif isinstance(image, torch.Tensor):
        image = transforms.ToPILImage()(image)

    image = transform_test(image).unsqueeze(0)
    image = image.to(device) # move image to the same device as model
    output = model(image) # output is a tensor of shape (1, num_classes)
    _, predicted = torch.max(output.data, 1) 
    return predicted.item()
    
# test on the first image in testset
first_image = testset[0][0] # get the first image from the testset
print(f'First image path: {testset.labels.iloc[0, 0]}')
classify_image(first_image)
print(f'Model prediction: {classify_image(first_image)}')
print(f'Reminder of the class labels: {classes}')

First image path: ../data/processed/Testing/no_tumor/image(103).jpg
Model prediction: 2
Reminder of the class labels: ['no_tumor', 'glioma_tumor', 'meningioma_tumor', 'pituitary_tumor']


/Users/zepyoorkhechadoorian/Documents/projects/mri-brain-tumor/.venv/lib/python3.8/site-packages/torchvision/transforms/functional.py:282: RuntimeWarning: invalid value encountered in cast
  npimg = (npimg * 255).astype(np.uint8)


# Retraining

In [12]:
num_epochs = 3  # Start with a small number for testing
train_losses, train_accs, test_accs = [], [], []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    for inputs, labels in trainloader:
        # Convert labels to integer indices if needed
        if isinstance(labels[0], str):
            labels = torch.tensor([classes.index(lbl) for lbl in labels])
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    train_loss = running_loss / len(trainloader.dataset)
    train_acc = 100. * correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # Evaluate on test set
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in testloader:
            if isinstance(labels[0], str):
                labels = torch.tensor([classes.index(lbl) for lbl in labels])
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    test_acc = 100. * correct / total
    test_accs.append(test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

Epoch [1/3] Train Loss: 2.5153 | Train Acc: 41.11% | Test Acc: 39.18%
Epoch [2/3] Train Loss: 1.0962 | Train Acc: 60.32% | Test Acc: 52.13%
Epoch [3/3] Train Loss: 0.9586 | Train Acc: 64.45% | Test Acc: 55.54%


# Model Evaluation

# Summary